# Fine-Tuning a Small Model for Financial Brief Sections (QLoRA)

Fine-tunes **Qwen2.5-1.5B-Instruct** (Apache-2.0) with QLoRA to act as a drop-in
replacement for Claude Haiku on **2 of the 4** parallel section generators in
`agent/core.py`: **Financial Health** and **Risk Factors**.

SEC Filing Highlights and Recent Developments stay with Haiku — deterministic,
Claude-free targets could not be built for them (MD&A figures are table-bound;
NewsAPI coverage is too sparse). Sonnet synthesis is untouched.

**Pipeline:** raw harvest (done locally, `scripts/build_raw_data.py`) →
deterministic Claude-free target construction (inline below) → review 5 samples
(**halts**) → QLoRA train (gradient checkpointing, per-epoch checkpoints, loss
curve) → adapter to `/adapters/financial-lora/`.

Runs on a free Colab **T4**. Runtime → Change runtime type → T4 GPU.

## 1. Setup

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" \
    "accelerate>=0.33" "bitsandbytes>=0.43" matplotlib

import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Load the raw, Claude-free research data

`data/raw_research.jsonl` is produced locally by `scripts/build_raw_data.py`
(stock + raw 10-K Item 7 / Item 1A text). Upload it, or `git clone` the repo if
the data is committed.

In [ ]:
import os, json

RAW_PATH = "data/raw_research.jsonl"
if not os.path.exists(RAW_PATH):
    try:
        from google.colab import files
        print("Upload raw_research.jsonl:")
        up = files.upload()
        RAW_PATH = list(up.keys())[0]
    except Exception:
        raise FileNotFoundError("Provide data/raw_research.jsonl (run scripts/build_raw_data.py locally).")

rows = [json.loads(l) for l in open(RAW_PATH, encoding="utf-8") if l.strip()]
print(f"Loaded {len(rows)} tickers")

## 3. Dataset construction pipeline (deterministic, Claude-free)

The builders below are the **same** functions used by `scripts/build_dataset.py`
(sliced in verbatim). Every target is built deterministically from the raw data
— no LLM is called, so the dataset is genuinely Claude-free. `LOCAL_SECTIONS`
selects the 2 trained sections (Financial Health, Risk Factors).

In [ ]:
import re
import json
import html

# The 4 (heading, instruction) pairs from agent/core.py, inlined so the notebook
# is self-contained. Only the 2 in LOCAL_SECTIONS are used for training.
_SECTIONS = [
    ("### Financial Health",
     "Key metrics: price, market cap, P/E ratio, revenue, profit margin. Brief financial assessment. 3-5 sentences."),
    ("### Recent Developments",
     "Summarize the most relevant news and what it means for investors. 3-5 sentences."),
    ("### SEC Filing Highlights",
     "Key takeaways from the most recent 10-K or 10-Q. 3-5 sentences."),
    ("### Risk Factors",
     "2-3 primary risks an investor should be aware of, as a bullet list."),
]


SEC_CONTEXT_CAP = 3500  # chars of raw filing text put into the input context


# ── number formatting ───────────────────────────────────────────────────────────

def _money(v) -> str | None:
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    a = abs(v)
    if a >= 1e12:
        return f"${v / 1e12:.2f} trillion"
    if a >= 1e9:
        return f"${v / 1e9:.1f} billion"
    if a >= 1e6:
        return f"${v / 1e6:.1f} million"
    return f"${v:,.0f}"


def _pct(v) -> str | None:
    """Format a fraction (0.27 -> '27.0%'). For ratios stored as fractions."""
    try:
        return f"{float(v) * 100:.1f}%"
    except (TypeError, ValueError):
        return None


def _yield_pct(v) -> str | None:
    """yfinance reports dividend_yield already in percent units (0.37 == 0.37%),
    so it must NOT be multiplied by 100 like profit_margin is."""
    try:
        return f"{float(v):.2f}%"
    except (TypeError, ValueError):
        return None


_PUNCT_MAP = {
    "’": "'", "‘": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "…": "...", " ": " ",
    "�": "",  # actual replacement char, if any
}


def _clean(text: str) -> str:
    """Normalize smart punctuation to ASCII and collapse whitespace so curly
    quotes / em-dashes / stray encodings don't leak into targets."""
    if not text:
        return ""
    text = html.unescape(text)  # decode &#160; &amp; etc. left over from filing HTML
    for bad, good in _PUNCT_MAP.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip()


def _num(v, suffix="") -> str | None:
    try:
        return f"{float(v):.1f}{suffix}"
    except (TypeError, ValueError):
        return None


# ── Builder 1: Financial Health (templated from real figures) ───────────────────

def build_financial_health(stock: dict, company: str, ticker: str) -> str | None:
    price = stock.get("current_price")
    mcap = _money(stock.get("market_cap"))
    revenue = _money(stock.get("revenue"))
    net_income = _money(stock.get("net_income"))
    margin = _pct(stock.get("profit_margin"))
    pe = _num(stock.get("pe_ratio"), "x")
    fpe = _num(stock.get("forward_pe"), "x")
    sector = stock.get("sector")

    if price is None or (mcap is None and revenue is None):
        return None  # not enough to say anything grounded

    # Valuation phrasing keyed off the P/E level (deterministic, not invented).
    try:
        pe_val = float(stock.get("pe_ratio"))
        valuation = ("a premium valuation" if pe_val >= 30
                     else "a moderate valuation" if pe_val >= 15
                     else "a value-oriented multiple")
    except (TypeError, ValueError):
        valuation = None

    clauses = []
    trades = f"{company} trades at ${float(price):.2f} per share"
    if sector:
        trades += f" in the {sector.lower()} sector"
    clauses.append(trades + ".")

    if mcap:
        val = f"It carries a market capitalization of {mcap}"
        if pe:
            val += f" and a P/E ratio of {pe}"
            if fpe:
                val += f" ({fpe} forward)"
        if valuation:
            val += f", {valuation}"
        clauses.append(val + ".")

    if revenue or net_income or margin:
        parts = []
        if revenue:
            parts.append(f"{revenue} in annual revenue")
        if net_income:
            parts.append(f"{net_income} in net income")
        prof = "The company reports " + " and ".join(parts) if parts else "The company operates"
        if margin:
            prof += f", a net profit margin of {margin}"
        clauses.append(prof + ".")

    # Rotate the closing emphasis deterministically for surface variety.
    variant = sum(ord(c) for c in ticker) % 3
    hi, lo = stock.get("week_52_high"), stock.get("week_52_low")
    if variant == 0 and hi and lo:
        clauses.append(
            f"Over the past year the stock has ranged between ${float(lo):.2f} and ${float(hi):.2f}.")
    elif variant == 1 and stock.get("dividend_yield"):
        dy = _yield_pct(stock.get("dividend_yield"))
        if dy:
            clauses.append(f"It currently offers a dividend yield of {dy}.")

    return "### Financial Health\n" + " ".join(clauses)


# ── Builder 2: Recent Developments (extractive from real news) ───────────────────

def _first_sentence(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    m = re.split(r"(?<=[.!?])\s+", text)
    return m[0].strip()


def build_recent_developments(news: list) -> str | None:
    articles = [a for a in (news or []) if isinstance(a, dict) and a.get("title")][:3]
    if not articles:
        return None

    a0 = articles[0]
    lead = f'Recent coverage includes "{_clean(a0["title"])}"'
    if a0.get("source"):
        lead += f' ({_clean(a0["source"])})'
    lead += "."
    desc = _clean(_first_sentence(a0.get("description")))
    if desc:
        lead += f" {desc}{'' if desc.endswith(('.', '!', '?')) else '.'}"

    extra = []
    for a in articles[1:]:
        t = _clean(a["title"])
        extra.append(f'"{t}" ({_clean(a["source"])})' if a.get("source") else f'"{t}"')
    body = f" Additional headlines: {' and '.join(extra)}." if extra else ""

    return "### Recent Developments\n" + lead + body


# ── Builder 3 & 4: extractive helpers over raw filing text ──────────────────────

# Legal/disclaimer/exhibit boilerplate to drop before ranking, so the targets
# carry business content rather than filing scaffolding.
_BOILERPLATE = (
    "forward-looking statement", "forward looking statement", "private securities litigation",
    "securities act", "exchange act", "incorporated by reference", "table of contents",
    "see item", "see part", "pursuant to", "subsidiaries", "jurisdiction",
    "certificate of formation", "certificate of incorporation", "anti-takeover",
    "annual report on form", "quarterly report on form", "this report", "cautionary",
    "undue reliance", "as defined in", "the sec", "guidance issued",
    "unless otherwise stated", "all information presented", "fiscal calendar",
    "accompanying notes", "in conjunction with", "should be read", "refer to the",
    "is not an indication", "the discussion of", "set forth below", "described below",
    # Descriptive / non-analytical sentences (company profile, products, channels)
    # that score on financial terms but carry no analytical value.
    "fiscal year is", "week period", "founded in", "develop and support",
    "sells its products", "resells", "directly to customers", "retail and online",
    "distribution channels", "our products include", "wholly-owned",
)


def _is_boilerplate(s: str) -> bool:
    sl = s.lower()
    if any(b in sl for b in _BOILERPLATE):
        return True
    # Page furniture flattened into the text: pipes, running form headers, and
    # embedded "Item 7A." / "Item 8" header/footer fragments.
    if "|" in s:
        return True
    if re.search(r"\bform\s*10-[kq]\b", sl):
        return True
    if re.search(r"\bitem\s*\d+[a-z]?[\.\s]", sl):
        return True
    if "discussion and analysis of financial condition" in sl:
        return True
    # Drop header-like fragments that are mostly Capitalized Words (no real sentence).
    words = s.split()
    if words and sum(w[0].isupper() for w in words if w[0].isalpha()) / len(words) > 0.7:
        return True
    return False


def _sentences(text: str) -> list[str]:
    text = _clean(text)
    raw = re.split(r"(?<=[.!?])\s+", text)
    out = []
    for s in raw:
        s = s.strip()
        # Strip leading page-number / "Part I" furniture (e.g. "1 Part I Supervision...").
        s = re.sub(r"^(\d+\s+)?(part\s+[ivx]+\s+)?", "", s, flags=re.I).strip()
        if not (60 <= len(s) <= 300):
            continue
        letters = sum(c.isalpha() for c in s)
        if letters / max(len(s), 1) < 0.6:   # skip tables / numeric noise
            continue
        if len(s.split()) < 8:
            continue
        if _is_boilerplate(s):
            continue
        out.append(s)
    return out


def clean_sentence_count(text: str | None, terms: set, min_score: int) -> int:
    """How many non-boilerplate sentences clear the term-score bar — the yield
    metric used to verify the filtering before scaling."""
    if not text:
        return 0
    return sum(1 for s in _sentences(text) if _score(s, terms) >= min_score)


_HIGHLIGHT_TERMS = {
    "revenue", "sales", "growth", "grew", "increased", "decreased", "margin",
    "operating", "income", "net", "billion", "million", "fiscal", "quarter",
    "segment", "demand", "products", "services", "customers", "cash", "earnings",
    "performance", "results", "year", "operations",
}

_RISK_TERMS = {
    "risk", "risks", "adversely", "adverse", "could", "decline", "fail",
    "failure", "competition", "competitive", "regulatory", "regulation",
    "litigation", "uncertain", "uncertainty", "volatility", "disrupt",
    "unable", "negative", "loss", "losses", "harm", "materially",
}


def _score(sentence: str, terms: set) -> int:
    words = re.findall(r"[a-z]+", sentence.lower())
    return sum(1 for w in words if w in terms)


def build_sec_highlights(sec_raw: dict) -> str | None:
    # Item 7 (MD&A) is where management discusses results; fall back to risk text.
    text = sec_raw.get("mda") or sec_raw.get("risk_factors")
    if not text:
        return None
    sents = _sentences(text)
    if not sents:
        return None
    # An analytical highlight is either rich in financial terms (>=3) or a
    # figure-bearing result sentence (a real $/% amount with >=1 term). A bare
    # year/count no longer counts: descriptive sentences (fiscal-year
    # definitions, founding statements, distribution channels, product listings)
    # carry few financial terms and no $/%, so they fail both gates — and the
    # named ones are also caught by _BOILERPLATE. NB: many 10-Ks keep figures in
    # tables (flattened to number-soup, dropped by _sentences), so a hard
    # figure-only gate would wrongly discard whole sections.
    def has_figure(s):
        return bool(re.search(r"\$\s?\d|\d+(?:\.\d+)?\s?%|\bpercent\b", s))

    def qualifies(s):
        sc = _score(s, _HIGHLIGHT_TERMS)
        return sc >= 3 or (has_figure(s) and sc >= 1)

    def rank(s):  # prefer figure-bearing, term-rich sentences
        return _score(s, _HIGHLIGHT_TERMS) + (2 if has_figure(s) else 0)

    cands = [(i, s) for i, s in enumerate(sents) if qualifies(s)]
    cands.sort(key=lambda p: rank(p[1]), reverse=True)
    top = cands[:4]
    if len(top) < 2:
        return None
    top.sort(key=lambda p: p[0])  # restore document order
    return "### SEC Filing Highlights\n" + " ".join(s for _, s in top)


# Modal/risk-framing words that mark a real risk statement (vs a product
# description). A sentence qualifies as a risk bullet if it is risk-term-dense
# (>=2) OR has >=1 risk term AND a modal — so declarative risks like "the
# markets are highly competitive" (score 2) stay, single-weak-term product
# descriptions like the AppleCare bullet ("...theft and loss...", score 1, no
# modal) are dropped, and single-term-with-modal risks recover coverage.
_RISK_MODALS = ("could", "may", "might", "would", "adversely", "adverse",
                "subject to", "risk", "uncertain", "materially", "fail")


def build_risk_factors(sec_raw: dict) -> str | None:
    # Item 1A (Risk Factors) region, already isolated during harvest.
    text = sec_raw.get("risk_factors")
    if not text:
        return None
    sents = _sentences(text)

    def is_risk(s):
        sc = _score(s, _RISK_TERMS)
        if sc >= 2:
            return True
        sl = s.lower()
        return sc >= 1 and any(m in sl for m in _RISK_MODALS)

    cands = [(i, s) for i, s in enumerate(sents) if is_risk(s)]
    cands.sort(key=lambda p: _score(p[1], _RISK_TERMS), reverse=True)
    top = cands[:3]
    if len(top) < 2:
        return None
    top.sort(key=lambda p: p[0])
    bullets = "\n".join(f"- {s}" for _, s in top)
    return "### Risk Factors\n" + bullets


_BUILDERS = {
    "### Financial Health":      lambda r: build_financial_health(r["stock"], r["company_name"], r["ticker"]),
    "### Recent Developments":   lambda r: build_recent_developments(r["news"]),
    "### SEC Filing Highlights": lambda r: build_sec_highlights(r["sec_raw"]),
    "### Risk Factors":          lambda r: build_risk_factors(r["sec_raw"]),
}

# Sections the fine-tuned local model is trained on and serves. Two sections are
# deliberately excluded and kept on Haiku because deterministic extraction can't
# build grounded targets for them:
#   - Recent Developments: NewsAPI coverage is too sparse (18%).
#   - SEC Filing Highlights: MD&A figures are table-bound (flatten to number-soup)
#     and the remaining prose is descriptive, not analytical.
# Keep this list in sync with LOCAL_SECTIONS in agent/tools/local_model.py.
LOCAL_SECTIONS = ["### Financial Health", "### Risk Factors"]


# ── input context (mirrors _data_context but with raw, Claude-free SEC text) ─────

def train_context(stock: dict, news: list, sec_raw: dict, cap: int = SEC_CONTEXT_CAP) -> str:
    # SEC portion = raw Item 7 (MD&A) + Item 1A (Risk Factors) excerpts, capped.
    # Claude-free and aligned with what the section builders draw from.
    sec = {}
    if sec_raw.get("mda"):
        sec["MD&A"] = sec_raw["mda"][:cap]
    if sec_raw.get("risk_factors"):
        sec["Risk Factors"] = sec_raw["risk_factors"][:cap]
    return f"Stock: {json.dumps(stock)}\nNews: {json.dumps(news)}\nSEC: {json.dumps(sec)}"


def section_prompt(heading: str, instruction: str, company: str, ticker: str, context: str) -> str:
    # Identical wording to agent.core._haiku_section so train/inference match.
    return (
        f"Write ONLY the '{heading}' section for a {company} ({ticker}) investment brief.\n"
        f"{instruction}\nStart with the markdown heading. Be concise.\n\nData:\n{context}"
    )


def build_examples(row: dict) -> list[dict]:
    context = train_context(row["stock"], row["news"], row["sec_raw"])
    instr = {h: i for h, i in _SECTIONS}
    examples = []
    for heading in LOCAL_SECTIONS:  # Recent Developments stays with Haiku
        target = _BUILDERS[heading](row)
        if not target:
            continue
        prompt = section_prompt(heading, instr[heading], row["company_name"], row["ticker"], context)
        examples.append({
            "ticker": row["ticker"],
            "section": heading,
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": target},
            ],
        })
    return examples


## 4. Build the dataset and inspect coverage + yield

In [ ]:
from collections import Counter

examples = [ex for row in rows for ex in build_examples(row)]
cnt = Counter(ex["section"] for ex in examples)
n = len(rows)
print(f"Built {len(examples)} examples from {n} tickers "
      f"(SEC Highlights + Recent Developments stay on Haiku).\n")
print("Per-section coverage:")
for h in LOCAL_SECTIONS:
    c = cnt.get(h, 0)
    print(f"  {h:<28} {c:>3}/{n}  ({c/n*100:.0f}%)")

risk = [clean_sentence_count(r["sec_raw"].get("risk_factors"), _RISK_TERMS, 2) for r in rows]
print(f"\nRisk Factors filtered-sentence yield: "
      f"mean={sum(risk)/len(risk):.1f}  nonzero={sum(1 for x in risk if x)}/{len(risk)}")

## 5. Review 5 sample rows  ⛔ training halts until you approve

In [ ]:
shown = []
for sec in LOCAL_SECTIONS:
    for ex in examples:
        if ex["section"] == sec:
            shown.append(ex); break
shown += [ex for ex in examples if ex not in shown][:3]

print("SCHEMA: {ticker, section, messages:[{role:user, content}, {role:assistant, content}]}\n")
for ex in shown[:5]:
    print("=" * 72)
    print(f"{ex['ticker']}  |  {ex['section']}")
    print("USER (input, truncated):", ex["messages"][0]["content"][:180].replace("\n", " "), "...")
    print("ASSISTANT (deterministic target):")
    print(ex["messages"][1]["content"])
    print()
print(">>> REVIEW THE SAMPLES. Training will NOT run until you set PROCEED_TO_TRAIN = True below. <<<")

## 6. QLoRA fine-tuning

**Base:** Qwen2.5-1.5B-Instruct · **Quant:** 4-bit NF4 (QLoRA) ·
**LoRA:** r=16, α=32, dropout=0.05, all linear projections ·
**Gradient checkpointing:** on · **Checkpoints:** every epoch ·
loss masked to the assistant turn only.

In [ ]:
PROCEED_TO_TRAIN = False  # set True after reviewing the 5 samples above
assert PROCEED_TO_TRAIN, "Review the samples, then set PROCEED_TO_TRAIN = True to start training."


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
import random
from datasets import Dataset

random.seed(0)
random.shuffle(examples)
n_val = max(1, int(len(examples) * 0.1))
val_raw, train_raw = examples[:n_val], examples[n_val:]
MAX_LEN = 2048

def encode(ex):
    msgs = ex["messages"]
    prompt_text = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False)
    p_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    f_ids = tokenizer(full_text, add_special_tokens=False).input_ids[:MAX_LEN]
    labels = list(f_ids)
    for i in range(min(len(p_ids), len(labels))):
        labels[i] = -100  # mask the prompt; train on the assistant turn only
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels}

train_ds = Dataset.from_list([encode(e) for e in train_raw])
val_ds = Dataset.from_list([encode(e) for e in val_raw])

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        d = m - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * d)
        out["attention_mask"].append(b["attention_mask"] + [0] * d)
        out["labels"].append(b["labels"] + [-100] * d)
    return {k: torch.tensor(v) for k, v in out.items()}

print(f"train={len(train_ds)}  val={len(val_ds)}")

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="/content/adapters/financial-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",          # per-epoch adapter checkpoints
    eval_strategy="epoch",          # older transformers: evaluation_strategy
    save_total_limit=4,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)
trainer.train()

## 7. Loss curve

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="eval loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("QLoRA training"); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 8. Save the adapter to `/adapters/financial-lora/`

In [ ]:
import os, shutil

OUT = "/content/adapters/financial-lora"
os.makedirs(OUT, exist_ok=True)
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print("Saved adapter:", os.listdir(OUT))

shutil.make_archive("/content/financial-lora", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/financial-lora.zip")
except Exception:
    print("Download /content/financial-lora.zip manually if not in Colab.")

## 9. Merge → GGUF → Ollama (for local serving)

Merge the adapter into the base weights, then convert to GGUF and register with
Ollama so the agent can call it locally (see `agent/tools/local_model.py`).

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, OUT).merge_and_unload()
merged.save_pretrained("/content/financial-lora-merged")
tokenizer.save_pretrained("/content/financial-lora-merged")
print("Merged model saved to /content/financial-lora-merged")

```bash
# On your machine (one-time), convert the merged HF model to GGUF and load in Ollama:
git clone https://github.com/ggerganov/llama.cpp && pip install -r llama.cpp/requirements.txt
python llama.cpp/convert_hf_to_gguf.py financial-lora-merged --outfile financial-lora.gguf --outtype q8_0
llama.cpp/llama-quantize financial-lora.gguf financial-lora-q4.gguf q4_K_M

# Modelfile:
#   FROM ./financial-lora-q4.gguf
ollama create financial-lora -f Modelfile
ollama run financial-lora "Write ONLY the '### Financial Health' section..."
```
Then set `USE_LOCAL_MODEL=true` and `LOCAL_MODEL_NAME=financial-lora` in `.env`.